# Bab 2 — Experimental Design and Configuration

Notebook pendamping Bab 2 Level 6.

Notebook ini mengikuti halaman bab di situs. Jalankan sel dari atas ke bawah.

## Learning Objectives

Setelah mempelajari bab ini, kamu mampu:

- Membedakan **desain eksperimen** dari **implementasi algoritma**.
- Menentukan tujuan, dataset, variabel, algoritma, dan metrik untuk sebuah eksperimen.
- Menyusun konfigurasi baseline dan tabel konfigurasi eksperimen.
- Menerapkan prinsip "ubah satu variabel pada satu waktu" dan mengenali kapan itu tidak cukup.
- Menjalankan eksperimen multi-faktor sederhana dengan `ParameterGrid`.
- Mencatat pengaturan eksperimen (ID, data, praproses, parameter, seed, hasil, waktu, versi pustaka) ke CSV dan JSON.
- Memeriksa bahwa sebuah eksperimen bisa diulang.
- Membaca hasil desain eksperimen untuk K-Means, DBSCAN, dan Hierarchical Clustering.

## Langkah 0: Pindah ke folder utama proyek

In [ ]:
import os
import sys
from pathlib import Path

if not Path("data").exists():
    os.chdir("../..")

# alat bantu pencatat eksperimen (examples/level-6/_experiment.py)
sys.path.insert(0, str(Path("examples/level-6").resolve()))

## Bagian A: Siapkan alat

In [ ]:
import json

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.cluster import DBSCAN, AgglomerativeClustering, KMeans
from sklearn.datasets import make_blobs, make_moons
from sklearn.metrics import adjusted_rand_score
from sklearn.model_selection import ParameterGrid
from sklearn.preprocessing import StandardScaler

import _experiment as ex   # alat bantu pencatat (examples/level-6/_experiment.py)

## Bagian B: Dua dataset dan sidik jarinya

In [ ]:
SEED = 42
X_blobs_raw, y_blobs = make_blobs(n_samples=300, centers=3, cluster_std=1.2, random_state=SEED)
X_moons_raw, y_moons = make_moons(n_samples=300, noise=0.06, random_state=SEED)

# Praproses yang sama untuk semua eksperimen: StandardScaler
X_blobs = StandardScaler().fit_transform(X_blobs_raw)
X_moons = StandardScaler().fit_transform(X_moons_raw)

print("Blobs : bentuk", X_blobs.shape, "sidik jari data", ex.dataset_fingerprint(X_blobs))
print("Moons : bentuk", X_moons.shape, "sidik jari data", ex.dataset_fingerprint(X_moons))

## Bagian C: Satu fungsi untuk menjalankan satu konfigurasi

In [ ]:
def build_estimator(algorithm, params, seed):
    """Membuat estimator dari nama algoritma dan dictionary parameter."""
    if algorithm == "KMeans":
        return KMeans(n_init=10, random_state=seed, **params)
    if algorithm == "DBSCAN":
        return DBSCAN(**params)
    if algorithm == "Hierarchical":
        return AgglomerativeClustering(**params)
    raise ValueError(f"Algoritma tidak dikenal: {algorithm}")


def run_config(exp_id, dataset, X, y_ref, algorithm, params, seed=SEED):
    """Menjalankan satu konfigurasi dan mengembalikan satu catatan lengkap."""
    estimator = build_estimator(algorithm, params, seed)
    labels, seconds = ex.timed(estimator.fit_predict, X)       # pelatihan: hanya X
    scores = ex.internal_scores(X, labels)                      # metrik internal
    scores["ari_reference"] = adjusted_rand_score(y_ref, labels)  # label acuan: hanya untuk menilai
    return ex.make_record(exp_id, dataset, X, "StandardScaler", "semua fitur", algorithm, params, seed, scores, seconds)


columns = ["experiment_id", "n_clusters", "noise_fraction", "silhouette", "davies_bouldin", "ari_reference"]

## Bagian D: Desain K-Means: ubah K

In [ ]:
# Desain 1: K-Means pada blobs. Variabel bebas: K. Baseline: K = 3. Lainnya ditahan sama.
records = []
for i, k in enumerate([2, 3, 4, 5], start=1):
    records.append(run_config(f"KM-{i:02d}", "blobs", X_blobs, y_blobs, "KMeans", {"n_clusters": k}))
kmeans_table = pd.DataFrame(records)
print(kmeans_table[["experiment_id", "params"] + columns[1:]].round(3).to_string(index=False))

## Bagian E: Desain DBSCAN: ubah satu variabel

In [ ]:
# Desain 2: DBSCAN pada moons, satu variabel per kali. Baseline: eps = 0.3, min_samples = 5.
dbscan_records = []
for i, eps in enumerate([0.15, 0.2, 0.3, 0.4], start=1):          # variabel bebas 1: eps
    dbscan_records.append(run_config(f"DB-E{i}", "moons", X_moons, y_moons, "DBSCAN", {"eps": eps, "min_samples": 5}))
for i, min_samples in enumerate([3, 10, 20], start=1):             # variabel bebas 2: min_samples (eps tetap 0.3)
    dbscan_records.append(run_config(f"DB-M{i}", "moons", X_moons, y_moons, "DBSCAN", {"eps": 0.3, "min_samples": min_samples}))
dbscan_table = pd.DataFrame(dbscan_records)
print(dbscan_table[["experiment_id", "params"] + columns[1:]].round(3).to_string(index=False))

## Bagian F: Desain Hierarchical: ubah linkage

In [ ]:
# Desain 3: Hierarchical pada moons. Variabel bebas: linkage. Baseline: ward. n_clusters = 2 tetap.
hier_records = []
for i, linkage in enumerate(["ward", "complete", "average", "single"], start=1):
    hier_records.append(run_config(f"HC-{i:02d}", "moons", X_moons, y_moons, "Hierarchical", {"n_clusters": 2, "linkage": linkage}))
hier_table = pd.DataFrame(hier_records)
print(hier_table[["experiment_id", "params"] + columns[1:]].round(3).to_string(index=False))

## Bagian G: Pengantar multi-faktor: eps x min_samples

In [ ]:
# Pengantar multi-faktor: dua variabel diubah bersama, semua kombinasinya dijalankan
grid = ParameterGrid({"eps": [0.2, 0.3, 0.4], "min_samples": [3, 5, 10]})
factorial_records = []
for i, params in enumerate(grid, start=1):
    factorial_records.append(run_config(f"DB-F{i:02d}", "moons", X_moons, y_moons, "DBSCAN", params))
factorial_table = pd.DataFrame(factorial_records)

# Kolom params berisi teks JSON. Kita urai supaya eps dan min_samples bisa dijadikan tabel silang
factorial_table["eps"] = [json.loads(text)["eps"] for text in factorial_table["params"]]
factorial_table["min_samples"] = [json.loads(text)["min_samples"] for text in factorial_table["params"]]
print("Jumlah kelompok (baris = eps, kolom = min_samples):")
print(factorial_table.pivot(index="eps", columns="min_samples", values="n_clusters").to_string())
print("Proporsi noise:")
print(factorial_table.pivot(index="eps", columns="min_samples", values="noise_fraction").round(3).to_string())
print("ARI terhadap label acuan (dinilai SETELAH clustering):")
print(factorial_table.pivot(index="eps", columns="min_samples", values="ari_reference").round(3).to_string())

## Bagian H: Menyimpan hasil dan konfigurasi

In [ ]:
all_records = records + dbscan_records + hier_records + factorial_records
results = ex.save_records(all_records, "02_experiments")
ex.save_json(ex.environment_info(), "02_environment")
ex.save_json(
    {
        "seed": SEED,
        "datasets": {"blobs": "make_blobs(n_samples=300, centers=3, cluster_std=1.2)", "moons": "make_moons(n_samples=300, noise=0.06)"},
        "preprocessing": "StandardScaler",
        "designs": {"KM": "ubah n_clusters", "DB-E": "ubah eps", "DB-M": "ubah min_samples", "HC": "ubah linkage", "DB-F": "grid eps x min_samples"},
    },
    "02_config",
)
print("Jumlah catatan eksperimen :", len(results))
print("Kolom catatan             :", list(results.columns))
print("Info lingkungan           :", ex.environment_info())

## Bagian I: Memeriksa bahwa eksperimen bisa diulang

In [ ]:
# Jalankan ulang semua konfigurasi, lalu bandingkan dengan catatan tersimpan (waktu jalan tidak dibandingkan)
rerun = []
for rec in all_records:
    params = json.loads(rec["params"])
    dataset = X_blobs if rec["dataset"] == "blobs" else X_moons
    y_ref = y_blobs if rec["dataset"] == "blobs" else y_moons
    rerun.append(run_config(rec["experiment_id"], rec["dataset"], dataset, y_ref, rec["algorithm"], params, rec["seed"]))
rerun_table = pd.DataFrame(rerun).drop(columns="runtime_seconds")
saved_table = pd.read_csv(ex.RESULTS_DIR / "02_experiments.csv").drop(columns="runtime_seconds")
same = np.allclose(rerun_table[["n_clusters", "noise_fraction", "silhouette", "ari_reference"]].fillna(-9).to_numpy(float),
                   saved_table[["n_clusters", "noise_fraction", "silhouette", "ari_reference"]].fillna(-9).to_numpy(float))
print("Semua hasil identik dengan catatan tersimpan (tanpa waktu jalan):", bool(same))

## Bagian J: Menggambar kondisi DBSCAN

In [ ]:
palette = ["#0072B2", "#E69F00", "#009E73", "#CC79A7", "#D55E00", "#56B4E9"]

fig, axes = plt.subplots(1, 4, figsize=(16, 4))
for ax, eps in zip(axes, [0.15, 0.2, 0.3, 0.4]):
    labels = DBSCAN(eps=eps, min_samples=5).fit_predict(X_moons)
    for c in sorted(set(labels)):
        if c == -1:
            ax.scatter(X_moons[labels == c, 0], X_moons[labels == c, 1], s=22, marker="x", color="gray")
        else:
            ax.scatter(X_moons[labels == c, 0], X_moons[labels == c, 1], s=12, color=palette[c % len(palette)])
    ax.set_title(f"eps = {eps}  ({len(set(labels) - {-1})} kelompok, {int((labels == -1).sum())} noise)", fontsize=9)
    ax.set_xlabel("Fitur 1 (terskala)")
axes[0].set_ylabel("Fitur 2 (terskala)")
plt.tight_layout()
plt.show()

fig, axes = plt.subplots(1, 2, figsize=(10, 4))
for ax, (column, title) in zip(axes, [("n_clusters", "Jumlah kelompok"), ("noise_fraction", "Proporsi noise")]):
    pivot = factorial_table.pivot(index="eps", columns="min_samples", values=column)
    image = ax.imshow(pivot.to_numpy(float), cmap="viridis", aspect="auto")
    ax.set_xticks(range(len(pivot.columns)))
    ax.set_xticklabels(pivot.columns)
    ax.set_yticks(range(len(pivot.index)))
    ax.set_yticklabels(pivot.index)
    for r in range(pivot.shape[0]):
        for c in range(pivot.shape[1]):
            ax.text(c, r, f"{pivot.to_numpy(float)[r, c]:.2f}".rstrip("0").rstrip("."), ha="center", va="center", color="white", fontsize=9)
    ax.set_xlabel("min_samples")
    ax.set_ylabel("eps")
    ax.set_title(title)
    fig.colorbar(image, ax=ax, fraction=0.046)
plt.tight_layout()
plt.show()

## Latihan

1. Tambahkan konfigurasi DBSCAN `eps = 0,25`, `min_samples = 5` pada moons. Berapa kelompok, proporsi noise, dan ARI-nya?
2. Jalankan Hierarchical dengan `n_clusters = 3` dan `linkage = single` serta `ward` pada **blobs**. Berapa ARI dan ukuran kelompok `single`?
3. Buat blobs dengan `random_state=7` (bukan 42), standarkan, dan hitung sidik jarinya. Apa bedanya dengan sidik jari blobs asli?
4. Dari tabel multi-faktor, jelaskan mengapa desain OVAT dengan baseline `eps = 0,3` tidak akan menunjukkan bahwa `min_samples = 10` bisa berbahaya.
5. Mengapa DB-M3 tidak boleh dianggap konfigurasi terbaik walau Silhouette-nya tertinggi?
6. Tulis tabel konfigurasi (ID, dataset, praproses, algoritma, parameter, seed, peran) untuk desain K-Means K = 2 sampai 5.

Jawaban lengkap ada di halaman bab.

## Ringkasan

- **Desain eksperimen** adalah rencana tertulis yang dibuat sebelum menjalankan: tujuan, dataset, variabel, algoritma, metrik, baseline, dan cara mencatat. Ia berbeda dari implementasi algoritma (Level 4).
- **Satu variabel pada satu waktu** (OVAT) membuat sebab perbedaan jelas, tetapi tidak menunjukkan **interaksi**. Desain **multi-faktor** (`ParameterGrid`) mengujinya, dengan jumlah kondisi yang tumbuh cepat.
- Pada moons, hasil DBSCAN berubah drastis antara `eps` 0,2 dan 0,3, lalu stabil antara 0,3 dan 0,4. Efek `min_samples = 10` bergantung pada `eps` (23,3% noise pada `eps = 0,2`, 0,7% pada `eps = 0,3`).
- Konfigurasi dengan `min_samples = 20` tampak terbaik menurut Silhouette (0,617), tetapi 49,7% data jadi noise dan ARI hanya 0,196.
- Pada moons, `single` linkage menemukan dua bulan (ARI 1,000) tetapi punya metrik internal terburuk.
- Catat ID, sidik jari data, praproses, parameter, seed, metrik, waktu, dan versi pustaka dalam CSV dan JSON.
- Semua 24 konfigurasi dijalankan ulang dan identik dengan catatan (tanpa waktu jalan).